# 2D Allen-Cahn Visualization

Runs a single 2D Allen-Cahn simulation on the periodic torus $[0, L_x] \times [0, L_y]$ and visualises the field $u(x, y, t)$.

**Equation:**
$$ u_t = \Delta u + \frac{1}{\varepsilon^2}\,(u - u^3), \qquad (x, y) \in [0, L_x] \times [0, L_y], \text{ periodic.} $$

**Free energy / dynamics:**
$$ E[u] = \int \tfrac{1}{2}|\nabla u|^2 + \tfrac{1}{4\varepsilon^2}(u^2 - 1)^2 \, dx\,dy, \qquad u_t = -\frac{\delta E}{\delta u}. $$
Solutions evolve toward the bistable wells $u = \pm 1$ separated by sharp interfaces of thickness $O(\varepsilon)$ that subsequently coarsen by mean curvature flow.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rootutils
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

rootutils.setup_root(".", indicator=".project-root", pythonpath=True)
from data_generation.allen_cahn_2d.generate import InitialCondition2D, simulate_allen_cahn_2d  # noqa: E402

In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────────
EPS = 0.1
LX = 2 * np.pi
LY = 2 * np.pi
NX = 128
NY = 128
T = 0.2
DT = 4e-5
SAVE_EVERY = 100
NK = 5
MAX_K = 5

rng = np.random.default_rng(42)
u_ic = InitialCondition2D(LX, LY, Nk=NK, max_k=MAX_K)
u_ic.reset(rng)
print("IC:", u_ic)
print(f"eps = {EPS},  Lx = {LX:.3f},  Ly = {LY:.3f},  Nx = {NX},  Ny = {NY}")
print(f"T = {T},  dt = {DT:g},  save_every = {SAVE_EVERY}  (~{int(T / DT / SAVE_EVERY) + 1} frames)")

In [ ]:
# ── Run simulation ─────────────────────────────────────────────────────────────
t_coord, x_coord, y_coord, u_series = simulate_allen_cahn_2d(
    u_ic,
    eps=EPS,
    Lx=LX,
    Ly=LY,
    Nx=NX,
    Ny=NY,
    T=T,
    dt=DT,
    save_every=SAVE_EVERY,
)

print(f"u_series shape : {u_series.shape}  (frames, Nx, Ny)")
print(f"t range        : [{t_coord[0]:.4f}, {t_coord[-1]:.4f}]")
print(f"u range        : [{u_series.min():.4f}, {u_series.max():.4f}]")

In [ ]:
# ── Allen-Cahn double-well potential ──────────────────────────────────────────
u_plot = np.linspace(-1.5, 1.5, 400)
F_potential = 0.25 * (u_plot**2 - 1.0) ** 2 / EPS**2  # F(u) = (u^2 - 1)^2 / (4 eps^2)
f_force = (u_plot - u_plot**3) / EPS**2  # -F'(u)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(u_plot, F_potential, lw=2)
axes[0].axvline(-1, color="gray", ls=":")
axes[0].axvline(1, color="gray", ls=":")
axes[0].set_xlabel(r"$u$")
axes[0].set_ylabel(r"$F(u) = (u^2-1)^2 / (4\varepsilon^2)$")
axes[0].set_title(rf"Double-well potential ($\varepsilon$ = {EPS})")

axes[1].plot(u_plot, f_force, lw=2)
axes[1].axhline(0, color="gray", ls=":")
axes[1].set_xlabel(r"$u$")
axes[1].set_ylabel(r"$(u - u^3)/\varepsilon^2$")
axes[1].set_title("Reactive forcing")
fig.tight_layout()
plt.show()

In [ ]:
# ── Static snapshots: initial / quarter / half / final ────────────────────────
n_frames = u_series.shape[0]
snap_idx = [0, n_frames // 4, n_frames // 2, n_frames - 1]

fig, axes = plt.subplots(1, len(snap_idx), figsize=(4 * len(snap_idx), 4))
for ax, idx in zip(axes, snap_idx):
    im = ax.imshow(
        u_series[idx].T,
        origin="lower",
        extent=[x_coord[0], x_coord[-1], y_coord[0], y_coord[-1]],
        cmap="RdBu_r",
        vmin=-1.0,
        vmax=1.0,
    )
    ax.set_title(rf"$t$ = {t_coord[idx]:.4f}")
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")
    ax.set_aspect("equal")
fig.colorbar(im, ax=axes.tolist(), shrink=0.8, label=r"$u$")
fig.suptitle(rf"2D Allen-Cahn: $u(x, y, t)$  ($\varepsilon$ = {EPS})", y=1.02)
plt.show()

In [ ]:
# ── Animation ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(
    u_series[0].T,
    origin="lower",
    extent=[x_coord[0], x_coord[-1], y_coord[0], y_coord[-1]],
    cmap="RdBu_r",
    vmin=-1.0,
    vmax=1.0,
    animated=True,
)
title = ax.set_title(rf"$t$ = {t_coord[0]:.4f}")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_aspect("equal")
fig.colorbar(im, ax=ax, label=r"$u$")
fig.tight_layout()


def update(frame):
    im.set_array(u_series[frame].T)
    title.set_text(rf"$t$ = {t_coord[frame]:.4f}")
    return im, title


ani = FuncAnimation(fig, update, frames=n_frames, interval=80, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
# ── Diagnostics: mean, std, free-energy estimate over time ────────────────────
dx = LX / NX
dy = LY / NY

u_mean = u_series.mean(axis=(1, 2))
u_std = u_series.std(axis=(1, 2))


# Free energy: E[u] = sum( 0.5 |grad u|^2 + (u^2-1)^2 / (4 eps^2) ) * dx * dy
# Use periodic centered differences via np.roll.
def free_energy(u: np.ndarray) -> float:
    ux = (np.roll(u, -1, axis=0) - np.roll(u, 1, axis=0)) / (2 * dx)
    uy = (np.roll(u, -1, axis=1) - np.roll(u, 1, axis=1)) / (2 * dy)
    grad_sq = ux**2 + uy**2
    well = (u**2 - 1.0) ** 2 / (4.0 * EPS**2)
    return float(np.sum(0.5 * grad_sq + well) * dx * dy)


energy = np.array([free_energy(u_series[i]) for i in range(n_frames)])

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].plot(t_coord, u_mean, lw=1.5)
axes[0].set_xlabel(r"$t$")
axes[0].set_ylabel(r"$\langle u \rangle$")
axes[0].set_title("Spatial mean")
axes[0].grid(alpha=0.3)

axes[1].plot(t_coord, u_std, lw=1.5)
axes[1].set_xlabel(r"$t$")
axes[1].set_ylabel(r"$\mathrm{std}(u)$")
axes[1].set_title("Spatial std (\u2192 1 as system phase-separates)")
axes[1].grid(alpha=0.3)

axes[2].plot(t_coord, energy, lw=1.5, color="C2")
axes[2].set_xlabel(r"$t$")
axes[2].set_ylabel(r"$E[u]$")
axes[2].set_title("Free energy (monotone decreasing)")
axes[2].grid(alpha=0.3)
fig.tight_layout()
plt.show()